In [1]:
from envs.robot_arm_env import RobotArmEnvironment
from gymnasium.envs.registration import register
import xml.etree.ElementTree as ET
from time import sleep
from random import randint, choice, shuffle
import plotly.graph_objects as go

import numpy as np

AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


In [2]:
register(id="PhysicalFetchPickAndPlaceDense-v0",
         entry_point="envs.physical_fetch_env:PhysicalObstacleFetchPickAndPlaceEnv",
         kwargs={"reward_type": "dense"})

In [3]:
def get_positions_from_xy(x, y):
    return round(1.075 + x * 0.05, 5), round(0.525 + y * 0.05, 5)

In [4]:
def put_obstacle(x, y):
    assert 0 <= x <= 9
    assert 0 <= y <= 9

    tree = ET.parse('./assets/fetch/pick_and_place_obstacles.xml')
    root = tree.findall("worldbody")[0]
    obstacle = None
    for node in root.findall("body"):
        if node.get("name") == "obstacle_1":
            obstacle = node
            break
    x, y = get_positions_from_xy(x, y)
    obstacle.set("pos", f"{x} {y} {0.425}")
    tree.write("./assets/fetch/pick_and_place_obstacles.xml")

In [5]:
def set_goal_position(x, y):
    with open("./goal.txt", "w") as f:
        x, y = get_positions_from_xy(x, y)
        f.write(f"{x} {y} {0.425}\n")

In [6]:
def set_target_position(x, y):
    with open("./target.txt", "w") as f:
        x, y = get_positions_from_xy(x, y)
        f.write(f"{x} {y} {0.425}\n")

In [7]:
class RobotArmActor:
    def __init__(self, robot_env=None, state=None):
        self.robot_env = robot_env
        self.xyz = [0, 0, 0]
        self.g = 0
        self.Q = np.zeros((10, 10, 10, 10, 10, 10, 2, 10, 10, 2, 2), dtype=np.float32)
        self.state = state

    def move_to(self, x, y, z, g):
        dt = 0.04
        for i in range(10):
            vx, vy, vz = x - self.xyz[0], y - self.xyz[1], z - self.xyz[2]

            vg = (g - self.g) / 4

            if dt > abs(vx):
                vx = vx / dt
            else:
                vx = abs(vx) / vx

            if dt > abs(vz):
                vz = vz / dt
            else:
                vz = abs(vz) / vz

            if dt > abs(vy):
                vy = vy / dt
            else:
                vy = abs(vy) / vy

            new_position, _, _, _, _, _ = self.robot_env.step([vx, vy, vz, vg])
            self.xyz = new_position["observation"][:3]
            self.g = new_position["observation"][9] + new_position["observation"][10]
        self.g = g

    def start_from(self, xr, yr, zr, gr):
        x, y = get_positions_from_xy(xr, yr)
        z = 0.425 + zr * 0.05 + 0.03 * (1 - gr)
        g = 0

        if gr:
            g = 0.05

        self.move_to(x, y, z + 0.07, 0)
        self.move_to(x, y, z + 0.07, g)
        self.move_to(x, y, z, g)

    def to_xyzg(self, xr, yr, zr, gr):
        x, y = get_positions_from_xy(xr, yr)
        z = 0.425 + zr * 0.05
        g = 0

        if gr:
            g = 0.05

        self.move_to(x, y, z, g)

    @classmethod
    def get_random_state(cls):
        possible_states = {(i, j) for i in range(10) for j in range(10)}
        xg, yg = choice(list(possible_states))

        xr, yr = choice(list(possible_states))
        zr = randint(0, 1)
        possible_states.remove((xg, yg))

        xo, yo = choice(list(possible_states))
        possible_states.remove((xo, yo))

        xt, yt = choice(list(possible_states))
        zt = 0

        if zr == 0 and ((xr == xo and abs(yr - yo) <= 1) or (xr == xt and abs(yr - yt) <= 1)):
            gr = 1
        else:
            gr = randint(0, 1)

        return xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr

    @classmethod
    def get_actions(cls, state):
        actions = set()
        xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = state
        if zr == 0:
            actions.add("up")

        actions.add("down")
        if zr == 0:
            actions.remove("down")
        elif gr == 0 and (xr == xt and yr == yt and zt == 0 or xr == xo and yr == yo):
            actions.remove("down")
        elif gr == 1 and (xr == xt and abs(yr - yt) == 1 and zt == 0 or xr == xo and abs(yr - yo) == 1):
            actions.remove("down")

        actions.add("ungrab")
        if gr == 1:
            actions.remove("ungrab")
        elif zr == 0:
            if (xr == xo and abs(yr - yo) == 1) or (xr == xt and abs(yr - yt) == 1):
                actions.remove("ungrab")
        else:
            if xr == xt and yr == yt and zr == zt:
                actions.remove("ungrab")

        actions.add("grab")
        if gr == 0:
            actions.remove("grab")
        elif zr == 0 and xr == xo and yr == yo:
            actions.remove("grab")

        actions.add("forward")
        if xr == 9:
            actions.remove("forward")
        elif zr == 0 and gr == 0:
            if abs(yr - yo) <= 1 and xr + 1 == xo:
                actions.remove("forward")
            elif yr == yt and xr + 1 == xt and (xr == 8 or yr == yo and xr + 2 == xo):
                actions.remove("forward")
        elif zr == 0 and gr == 1:
            if abs(yr - yt) == 1 and xr + 1 == xt or abs(yr - yo) == 1 and xr + 1 == xo:
                actions.remove("forward")

        actions.add("backward")
        if xr == 0:
            actions.remove("backward")
        elif zr == 0 and gr == 0:
            if abs(yr - yo) <= 1 and xr - 1 == xo:
                actions.remove("backward")
            elif yr == yt and xr - 1 == xt and (xr == 1 or yr == yo and xr - 2 == xo):
                actions.remove("backward")
        elif zr == 0 and gr == 1:
            if abs(yr - yt) <= 1 and xr - 1 == xt or abs(yr - yo) <= 1 and xr - 1 == xo:
                actions.remove("backward")

        actions.add("left")
        if yr == 0:
            actions.remove("left")
        elif zr == 0:
            if gr == 0 and (xr == xt and yr - 1 == yt or xr == xo and yr - 1 == yo):
                actions.remove("left")
            elif gr == 1 and (xr == xt and abs(yr - yt) <= 2 or xr == xo and abs(yr - yo) <= 2):
                actions.remove("left")

        actions.add("right")
        if yr == 9:
            actions.remove("right")
        elif zr == 0:
            if gr == 0 and (xr == xt and yr + 1 == yt or xr == xo and yr + 1 == yo):
                actions.remove("right")
            elif gr == 1 and (xr == xt and abs(yr - yt) <= 2 or xr == xo and abs(yr - yo) <= 2):
                actions.remove("right")
        actions = list(actions)
        shuffle(actions)
        return actions

    def set_state(self, state=None):
        if state is None:
            state = RobotArmActor.get_random_state()
        xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = state

        self.state = (xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr)

        put_obstacle(xo, yo)
        set_goal_position(xg, yg)
        set_target_position(xt, yt)

        self.start_from(xr, yr, zr, gr)

    @classmethod
    def check_win(cls, state):
        if state[0] == state[4] and state[1] == state[5] and state[6] == 0:
            return True
        return False

    @classmethod
    def get_next_state(cls, state, action):
        xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = state
        replicate = False
        if xr == xt and yr == yt and zr == zt and (gr == 0 or gr == 1 and (action != "forward" and action != "backward" and action != "up")):
            replicate = True

        if action == "up":
            zr += 1
        elif action == "down":
            zr -= 1
        elif action == "left":
            yr -= 1
        elif action == "right":
            yr += 1
        elif action == "forward":
            xr += 1
            if xr == xt and yr == yt and zr == 0 and gr == 0:
                xt += 1
        elif action == "backward":
            xr -= 1
            if xr == xt and yr == yt and zr == 0 and gr == 0:
                xt -= 1
        elif action == "grab":
            gr = 0
        elif action == "ungrab":
            gr = 1

        if replicate:
            xt, yt, zt = xr, yr, zr

        return xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr

    def act(self, state, action):
        new_state = RobotArmActor.get_next_state(state, action)
        return new_state

    def env_act(self):
        sleep(0.2)
        action = self.choose_best_action(self.state, RobotArmActor.get_actions(self.state))
        # print(self.state, action)
        self.state = RobotArmActor.get_next_state(self.state, action)
        xr, yr, zr, gr = self.state[-4:]

        x, y = get_positions_from_xy(xr, yr)
        z = 0.425 + zr * 0.05 + (1 - gr) * 0.03
        g = 0

        if gr:
            g = 0.05

        if action == "grab":
            self.move_to(x, y, z - 0.03, g)
            self.move_to(x, y, z, g)
        elif action == "ungrab":
            self.move_to(x, y, z, 0)
            self.move_to(x, y, z, g)
        else:
            self.move_to(x, y, z, g)

        return self.get_reward_from_state(self.state), RobotArmActor.check_win(self.state)

    def get_reward_from_state(self, state):
        xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = state
        try:
            return self.Q[xg][yg][xo][yo][xt][yt][zt][xr][yr][zr][gr]
        except Exception as e:
            print(state)
            raise e

    def set_reward_for_state(self, state, reward):
        xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = state
        self.Q[xg][yg][xo][yo][xt][yt][zt][xr][yr][zr][gr] = reward

    def choose_best_action(self, state, actions):
        rewards = []
        for action in actions:
            rewards.append(self.get_reward_from_state(RobotArmActor.get_next_state(state, action)))
        return actions[rewards.index(max(rewards))]

    def get_action_epsilon(self, state, actions):
        if randint(1, 100) <= 5:
            return choice(actions)

        return self.choose_best_action(state, actions)

    def train(self, number_of_episodes, state, print_log=False):
        alpha = 0.05
        gamma = 0.95

        rewards_store = []
        steps_store = []
        for i in range(number_of_episodes):
            max_reward = -1000
            number_of_steps = 0
            n = 100
            S = state
            A = self.get_action_epsilon(S, self.get_actions(S))
            while n and not RobotArmActor.check_win(S):
                n -= 1
                nS = self.act(S, A)
                nA = self.get_action_epsilon(nS, self.get_actions(nS))
                if print_log:
                    print(nS, nA)

                xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = S
                reward = -(abs(xg - xt) + abs(yg - yt) + zt + abs(xt - xr) + abs(yt - yr) + abs(zt - zr) + 1)

                if RobotArmActor.check_win(nS):
                    self.set_reward_for_state(nS, 1000)
                number_of_steps += 1
                self.set_reward_for_state(S, (1 - alpha) * self.get_reward_from_state(S) + alpha * (gamma * self.get_reward_from_state(nS) + reward))
                max_reward = max(max_reward, self.get_reward_from_state(S))
                A, S = nA, nS
            rewards_store.append(max_reward)
            steps_store.append(number_of_steps)

        return rewards_store, steps_store


In [8]:
ra = RobotArmActor()

In [9]:
RobotArmActor.get_random_state()

(3, 0, 0, 4, 8, 4, 0, 0, 2, 0, 1)

In [ ]:
xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr = 0, 0, 5, 5, 9, 9, 0, 5, 7, 0, 0
state = xg, yg, xo, yo, xt, yt, zt, xr, yr, zr, gr

In [37]:
ra.train(1000, state, print_log=False)
""

''

In [38]:
ra.train(1, state, print_log=True)

(0, 0, 5, 5, 9, 9, 0, 5, 8, 0, 0) forward
(0, 0, 5, 5, 9, 9, 0, 6, 8, 0, 0) forward
(0, 0, 5, 5, 9, 9, 0, 7, 8, 0, 0) forward
(0, 0, 5, 5, 9, 9, 0, 8, 8, 0, 0) ungrab
(0, 0, 5, 5, 9, 9, 0, 8, 8, 0, 1) right
(0, 0, 5, 5, 9, 9, 0, 8, 9, 0, 1) forward
(0, 0, 5, 5, 9, 9, 0, 9, 9, 0, 1) grab
(0, 0, 5, 5, 9, 9, 0, 9, 9, 0, 0) backward
(0, 0, 5, 5, 8, 9, 0, 8, 9, 0, 0) backward
(0, 0, 5, 5, 7, 9, 0, 7, 9, 0, 0) backward
(0, 0, 5, 5, 6, 9, 0, 6, 9, 0, 0) forward
(0, 0, 5, 5, 7, 9, 0, 7, 9, 0, 0) backward
(0, 0, 5, 5, 6, 9, 0, 6, 9, 0, 0) backward
(0, 0, 5, 5, 5, 9, 0, 5, 9, 0, 0) backward
(0, 0, 5, 5, 4, 9, 0, 4, 9, 0, 0) backward
(0, 0, 5, 5, 3, 9, 0, 3, 9, 0, 0) left
(0, 0, 5, 5, 3, 8, 0, 3, 8, 0, 0) left
(0, 0, 5, 5, 3, 7, 0, 3, 7, 0, 0) left
(0, 0, 5, 5, 3, 6, 0, 3, 6, 0, 0) backward
(0, 0, 5, 5, 2, 6, 0, 2, 6, 0, 0) left
(0, 0, 5, 5, 2, 5, 0, 2, 5, 0, 0) left
(0, 0, 5, 5, 2, 4, 0, 2, 4, 0, 0) left
(0, 0, 5, 5, 2, 3, 0, 2, 3, 0, 0) backward
(0, 0, 5, 5, 1, 3, 0, 1, 3, 0, 0) left
(0, 0, 5, 

([np.float32(915.8212)], [28])

In [39]:
robot_env = RobotArmEnvironment(env_name="PhysicalFetchPickAndPlaceDense-v0",
                                render_mode="human")
robot_env.setup()
robot_env.reset()

ra.robot_env = robot_env
ra.set_state(state)
finish = False
number_of_steps = 0
while not finish:
    number_of_steps += 1
    reward, finish = ra.env_act()
print("Spent steps:", number_of_steps, "Reward:", reward)
robot_env.close()

Spent steps: 26 Reward: 1000.0


In [25]:
rewards_accumulated = [0 for _ in range(1000)]
steps_accumulated = [0 for _ in range(1000)]

In [26]:
N = 100
for i in range(N):
    rewards, steps = ra.train(1000, RobotArmActor.get_random_state(), print_log=False)

    for i in range(1000):
        rewards_accumulated[i] += rewards[i] / N
        steps_accumulated[i] += steps[i] / N

In [27]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=rewards_accumulated))
fig.update_layout(title="Average reward during training", xaxis_title="Episode", yaxis_title="Reward")
fig.show()

In [28]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=steps_accumulated))
fig.update_layout(title="Average steps during training", xaxis_title="Episode", yaxis_title="Steps")
fig.show()

In [14]:
def run_for_state(state):
    robot_env = RobotArmEnvironment(env_name="PhysicalFetchPickAndPlaceDense-v0",
                                render_mode="human")
    robot_env.setup()
    robot_env.reset()
    ra = RobotArmActor()
    ra.robot_env = robot_env
    ra.set_state(state)
    rewards, steps = ra.train(1000, state, print_log=False)
    finish = False
    number_of_steps = 0
    while not finish:
        number_of_steps += 1
        reward, finish = ra.env_act()
    print("Spent steps:", number_of_steps, "Reward:", reward)
    robot_env.close()


    fig = go.Figure()
    fig.add_trace(go.Scatter(y=steps))
    fig.update_layout(title="Average steps during training", xaxis_title="Episode", yaxis_title="Steps")
    fig.show()

    fig = go.Figure()
    fig.add_trace(go.Scatter(y=rewards))
    fig.update_layout(title="Average reward during training", xaxis_title="Episode", yaxis_title="Reward")
    fig.show()

In [16]:
state = (9, 9, 9, 5, 9, 8, 0, 9, 8, 0, 1)
state = (4, 4, 5, 5, 6, 6, 0, 9, 9, 0, 0)
# state = (9, 9, 9, 5, 9, 7, 0, 9, 7, 0, 1)
state = (7, 5, 5, 5, 2, 5, 0, 9, 9, 0, 0)
run_for_state(state)

Spent steps: 22 Reward: 1000.0
